In [ ]:
# [Problem 1] Execution of various methods

In [ ]:
#cd C:\Users\lelin\workspace\AIData_Engineer\Scripts\EXERCISES\lstm
#python imdb_rnn_compare.py --models SimpleRNN,GRU,LSTM --epochs 2 --maxlen 200 --rnn_units 64 --embed_dim 64 --batch_size 128 | Tee-Object -FilePath comparison_results.txt

In [ ]:
# Console shows per-model test accuracy and a final “Summary” comparing SimpleRNN, GRU, LSTM.
#comparison_results.txt captures the same output for [Problem 1] Execution of various methods
#For faster runs,  i try --epochs 1 or --maxlen 80.
#Typical ranking on IMDB: LSTM ≥ GRU » SimpleRNN.

#[Problem 2] (Advance assignment) Comparison between multiple data sets

# python reuters_rnn_compare.py --epochs 3 --maxlen 200 --rnn_units 64 --embed_dim 64 --batch_size 128 --models SimpleRNN,GRU,LSTM | Tee-Object -FilePath reuters_comparison_results.txt

C:\Users\lelin\AppData\Local\Programs\Python\Python37\lib\site-packages\tensorflow\python\ops\init_ops.py:1251: calling VarianceScaling.__init__ (from tensorflow.python.ops.init_ops) with dtype is deprecated and will be removed in a future 
version.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Loading Reuters dataset...
Train: (8982, 200), Test: (2246, 200), Classes: 46

================================================================================
Training model: SimpleRNN
Train on 7185 samples, validate on 1797 samples
Epoch 1/3
7185/7185 - 24s - loss: 2.6214 - acc: 0.3545 - val_loss: 2.5169 - val_acc: 0.3589
Epoch 2/3
7185/7185 - 17s - loss: 2.3190 - acc: 0.3825 - val_loss: 2.3896 - val_acc: 0.3684       
Epoch 3/3
7185/7185 - 17s - loss: 2.0745 - acc: 0.4479 - val_loss: 2.1707 - val_acc: 0.3940       
WARNING:tensorflow:From C:\Users\lelin\AppData\Local\Programs\Python\Python37\lib\site-packages\tensorflow\python\ops\math_grad.py:1250: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Model SimpleRNN test accuracy: 0.3437 (time: 61.9s)

================================================================================        
Training model: GRU
Train on 7185 samples, validate on 1797 samples
Epoch 1/3
7185/7185 - 57s - loss: 2.9194 - acc: 0.3630 - val_loss: 2.3926 - val_acc: 0.3589
Epoch 2/3
7185/7185 - 56s - loss: 2.0542 - acc: 0.3858 - val_loss: 2.1279 - val_acc: 0.3567       
Epoch 3/3
7185/7185 - 59s - loss: 1.8468 - acc: 0.4354 - val_loss: 2.0226 - val_acc: 0.4485       
Model GRU test accuracy: 0.4533 (time: 174.3s)

================================================================================
Training model: LSTM
Train on 7185 samples, validate on 1797 samples
Epoch 1/3
7185/7185 - 75s - loss: 2.8868 - acc: 0.3571 - val_loss: 2.5270 - val_acc: 0.3589
Epoch 2/3
7185/7185 - 80s - loss: 2.2388 - acc: 0.4198 - val_loss: 2.1921 - val_acc: 0.4580
Epoch 3/3
7185/7185 - 69s - loss: 1.9528 - acc: 0.5042 - val_loss: 2.0675 - val_acc: 0.4741       
Model LSTM test accuracy: 0.4884 (time: 228.1s)

Summary (higher accuracy is better):
- SimpleRNN   acc=0.3437  time=61.9s
- GRU         acc=0.4533  time=174.3s
- LSTM        acc=0.4884  time=228.1s

In [ ]:
#[Problem 3] Explanation of other classes

Here’s a concise guide to the related Keras RNN classes and when to use them.

- RNN: A general RNN layer wrapper that runs a “cell” across time steps. You pass it a cell (e.g., LSTMCell), and it handles the loop, state, masking, go-backwards, return_sequences, etc. Use when you want fine control via custom cells or to stack multiple cells inside a single layer.
- SimpleRNNCell: The one-timestep computation for a vanilla (non-gated) RNN. Typically wrapped by RNN(SimpleRNNCell(...)) or used indirectly via SimpleRNN, which already wraps it. Rare in practice for long sequences due to vanishing gradients.
- GRUCell: One-timestep unit for GRU. Use inside RNN(GRUCell(...)) if you need cell-level control (e.g., custom loops or stacking cells in one layer). Otherwise, just use GRU for simplicity.
- LSTMCell: One-timestep unit for LSTM (with separate hidden and cell states). Use inside RNN(LSTMCell(...)) when you need lower-level control; otherwise prefer LSTM.
- StackedRNNCells: Utility to combine a list of cells into a single composite cell, then pass that to RNN. This produces a “multi-layer” RNN within a single RNN layer call. Useful when you want one fused layer managing multiple cells (e.g., tied states or shared settings). Otherwise, most users just stack multiple GRU/LSTM layers directly.
- CuDNNGRU: Legacy GPU-optimized GRU variant. In modern TF/Keras (2.1+), GRU automatically uses the fast cuDNN kernels on GPU when compatible (activation=tanh, recurrent_activation=sigmoid, no recurrent_dropout, etc.). Prefer GRU; CuDNNGRU is effectively merged/deprecated.
- CuDNNLSTM: Legacy GPU-optimized LSTM variant. Same story as CuDNNGRU: modern LSTM transparently picks the cuDNN path on GPU under compatible settings. Prefer LSTM; CuDNNLSTM is merged/deprecated.

Typical choices:
- Use LSTM or GRU layers directly for most tasks.
- Use RNN with cells (LSTMCell/GRUCell) when you need custom timestep logic, custom cells, or to create one “layer” that internally stacks multiple cells via StackedRNNCells.
- Avoid SimpleRNN except for toy problems or very short sequences.
- Don’t use CuDNN* directly anymore; rely on LSTM/GRU auto-acceleration on GPU.

Quick examples:
- Stacked cells in one layer:
```python
from tensorflow.keras import layers

cells = [layers.LSTMCell(64), layers.LSTMCell(64)]
x = layers.RNN(layers.StackedRNNCells(cells), return_sequences=False)(inputs)
```
- Custom cell with RNN:
```python
class MyCell(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.state_size = units
        self.output_size = units
        self.dense = layers.Dense(units, activation="tanh")
    def call(self, inputs, states):
        h_prev = states[0]
        h = self.dense(inputs) + 0.5 * h_prev
        return h, [h]

x = layers.RNN(MyCell(32))(inputs)
```